In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in the current working directory)
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Helper to compute summary for a dataframe with a name-column for KPI/feature
def make_summary(df, kpi_col):
    rows = len(df)
    min_ts = int(df['timestamp'].min()) if rows>0 else None
    max_ts = int(df['timestamp'].max()) if rows>0 else None
    # convert to timezone-aware datetimes in Asia/Shanghai (UTC+8)
    min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz) if min_ts is not None else None
    max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz) if max_ts is not None else None
    unique_cmdb = pd.Series(pd.unique(df['cmdb_id']))  # pandas Series for compact display
    unique_kpi = pd.Series(pd.unique(df[kpi_col])) if kpi_col in df.columns else pd.Series([], dtype=object)
    summary = pd.DataFrame([{
        'rows': rows,
        'min_timestamp': min_ts,
        'min_dt_UTC+8': min_dt,
        'max_timestamp': max_ts,
        'max_dt_UTC+8': max_dt,
        'unique_cmdb_count': unique_cmdb.size,
        'unique_kpi_count': unique_kpi.size
    }])
    # previews limited to 100 as required
    cmdb_preview = unique_cmdb.head(100).reset_index(drop=True).rename('cmdb_preview').to_frame()
    kpi_preview = unique_kpi.head(100).reset_index(drop=True).rename('kpi_preview').to_frame()
    return summary, cmdb_preview, kpi_preview

# Create summaries
metric_summary, metric_cmdb_preview, metric_kpi_preview = make_summary(metric, 'kpi_name')
trace_summary, trace_cmdb_preview, trace_kpi_preview = make_summary(trace, 'trace_name')
log_summary, log_cmdb_preview, log_kpi_preview = make_summary(log, 'log_name')

# Prepare error_logs sample (timestamp converted to UTC+8 datetime)
if len(error_logs) > 0:
    error_logs_sample = error_logs[['timestamp', 'cmdb_id', 'message']].sort_values('timestamp').head(10).copy()
    error_logs_sample['timestamp_dt_UTC+8'] = pd.to_datetime(error_logs_sample['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # keep columns compact and ordered
    error_logs_sample = error_logs_sample[['timestamp', 'timestamp_dt_UTC+8', 'cmdb_id', 'message']]
else:
    error_logs_sample = error_logs.head(0)

# Display compact summaries and previews (limited sizes)
metric_summary, metric_cmdb_preview.head(20), metric_kpi_preview.head(20), \
trace_summary, trace_cmdb_preview.head(20), trace_kpi_preview.head(20), \
log_summary, log_cmdb_preview.head(20), log_kpi_preview.head(20), \
error_logs_sample

```
Out[1]:
```
summary = (
    "metric.csv: 319,703 rows; timestamps 1647747000 → 1647750600 "
    "(2022-03-20 11:30:00+08 → 2022-03-20 12:30:00+08); "
    "59 unique cmdb_id; 1,368 unique kpi_name.\n"
    "trace.csv: 35,964 rows; timestamps 1647748800 → 1647750600 "
    "(2022-03-20 12:00:00+08 → 2022-03-20 12:30:00+08); "
    "40 unique cmdb_id; 200 unique trace_name.\n"
    "log.csv: 1,850 rows; timestamps 1647748800 → 1647750600 "
    "(2022-03-20 12:00:00+08 → 2022-03-20 12:30:00+08); "
    "31 unique cmdb_id; 2 unique log_name (log.error_count, log.row_count).\n"
    "error_logs.csv: 0 rows (empty) — no sample rows available."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     rows  min_timestamp              min_dt_UTC+8  max_timestamp              max_dt_UTC+8  unique_cmdb_count  unique_kpi_count
0  319703     1647747000 2022-03-20 11:30:00+08:00     1647750600 2022-03-20 12:30:00+08:00                 59              1368,           cmdb_preview
0            adservice
1          adservice-0
2          adservice-1
3          adservice-2
4           adservice2
5         adservice2-0
6          cartservice
7        cartservice-0
8        cartservice-1
9        cartservice-2
10      cartservice2-0
11     checkoutservice
12   checkoutservice-0
13   checkoutservice-1
14   checkoutservice-2
15  checkoutservice2-0
16     currencyservice
17   currencyservice-0
18   currencyservice-1
19   currencyservice-2,                                           kpi_preview
0                                      app.grpc.count
1                                        app.grpc.mrt
2                                         app.grpc.rr
3                                         app.grpc.sr
4                                      app.http.count
5                                        app.http.mrt
6                                         app.http.rr
7                                         app.http.sr
8     runtime.java_lang_ClassLoading_LoadedClassCount
9   runtime.java_lang_ClassLoading_TotalLoadedClas...
10  runtime.java_lang_ClassLoading_UnloadedClassCount
11             runtime.java_lang_ClassLoading_Verbose
12  runtime.java_lang_Compilation_CompilationTimeM...
13  runtime.java_lang_Compilation_TotalCompilation...
14  runtime.java_lang_GarbageCollector_CollectionC...
15  runtime.java_lang_GarbageCollector_CollectionC...
16  runtime.java_lang_GarbageCollector_CollectionT...
17  runtime.java_lang_GarbageCollector_CollectionT...
18  runtime.java_lang_GarbageCollector_LastGcInfo_...
19  runtime.java_lang_GarbageCollector_LastGcInfo_...,     rows  min_timestamp              min_dt_UTC+8  max_timestamp              max_dt_UTC+8  unique_cmdb_count  unique_kpi_count
0  35964     1647748800 2022-03-20 12:00:00+08:00     1647750600 2022-03-20 12:30:00+08:00                 40               200,           cmdb_preview
0          adservice-0
1          adservice-1
2          adservice-2
3         adservice2-0
4        cartservice-0
5        cartservice-1
6        cartservice-2
7       cartservice2-0
8    checkoutservice-0
9    checkoutservice-1
10   checkoutservice-2
11  checkoutservice2-0
12   currencyservice-0
13   currencyservice-1
14   currencyservice-2
15  currencyservice2-0
16      emailservice-0
17      emailservice-1
18      emailservice-2
19     emailservice2-0,                                    kpi_preview
0          trace.from_frontend-0.duration_mean
1           trace.from_frontend-0.duration_p95
2             trace.from_frontend-0.error_rate
3              trace.from_frontend-0.row_count
4          trace.from_frontend-1.duration_mean
5           trace.from_frontend-1.duration_p95
6             trace.from_frontend-1.error_rate
7              trace.from_frontend-1.row_count
8          trace.from_frontend-2.duration_mean
9           trace.from_frontend-2.duration_p95
10            trace.from_frontend-2.error_rate
11             trace.from_frontend-2.row_count
12        trace.from_frontend2-0.duration_mean
13         trace.from_frontend2-0.duration_p95
14           trace.from_frontend2-0.error_rate
15            trace.from_frontend2-0.row_count
16  trace.from_checkoutservice-0.duration_mean
17   trace.from_checkoutservice-0.duration_p95
18     trace.from_checkoutservice-0.error_rate
19      trace.from_checkoutservice-0.row_count,    rows  min_timestamp              min_dt_UTC+8  max_timestamp              max_dt_UTC+8  unique_cmdb_count  unique_kpi_count
0  1850     1647748800 2022-03-20 12:00:00+08:00     1647750600 2022-03-20 12:30:00+08:00                 31                 2,          cmdb_preview
0         adservice-0
1         adservice-1
2         adservice-2
3       cartservice-0
4       cartservice-1
5       cartservice-2
6   checkoutservice-0
7   checkoutservice-1
8   checkoutservice-2
9   currencyservice-0
10  currencyservice-1
11  currencyservice-2
12     emailservice-0
13     emailservice-1
14     emailservice-2
15         frontend-0
16         frontend-1
17         frontend-2
18   paymentservice-0
19   paymentservice-1,        kpi_preview
0  log.error_count
1    log.row_count, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing 'metric' variable if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (before any filtering)
global_p95 = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# total distinct component-KPIs scanned (from full metric file)
total_scanned = global_p95.shape[0]

# 2) Filter metric.csv rows to the incident window: 2022-03-20 12:00:00+08 to 2022-03-20 12:30:00+08 inclusive
start_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
end_ts = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 3) For each (cmdb_id, kpi_name) compute anomaly_count, earliest_anomaly_timestamp, max_value_in_window, and include global_P95
# Merge windowed data with global P95 thresholds
merged = metric_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value >= global_P95
merged['is_anomaly'] = merged['value'] >= merged['global_P95']

# Aggregations
anomalies = (
    merged[merged['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(earliest_anomaly_timestamp=('timestamp', 'min'),
         anomaly_count=('timestamp', 'count'))
    .reset_index()
)

max_vals = (
    merged
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(max_value_in_window=('value', 'max'))
    .reset_index()
)

# Combine results: ensure all groups that appear in the window are represented.
results = (
    max_vals
    .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anomalies, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill missing anomaly counts with 0 and keep earliest timestamp nullable
results['anomaly_count'] = results['anomaly_count'].fillna(0).astype(int)
results['earliest_anomaly_timestamp'] = results['earliest_anomaly_timestamp'].astype('Int64')  # nullable int
# reorder and select required columns
results = results[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'global_P95']]

# 4) Return compact table: top 20 by anomaly_count desc, and total number of distinct component-KPIs scanned
top20 = results.sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20)

top20, total_scanned

```
Out[2]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse kernel variables: top20, total_scanned
top = top20.copy()

# analyze top20 entries
all_counts_equal = top['anomaly_count'].nunique() == 1
common_count = int(top['anomaly_count'].iloc[0]) if all_counts_equal else None
earliest_ts_values = top['earliest_anomaly_timestamp'].dropna().unique()
earliest_dt = (pd.to_datetime(int(earliest_ts_values.min()), unit='s', utc=True)
               .tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')) if len(earliest_ts_values)>0 else None

# select example KPI and component names (up to 10)
examples = top[['cmdb_id', 'kpi_name']].head(10).to_records(index=False)
examples_list = [f"{c}/{k}" for c, k in examples]

summary = (
    f"Scanned {int(total_scanned)} distinct component-KPIs in metric.csv.\n"
    f"Incident window: 2022-03-20 12:00:00+08 → 2022-03-20 12:30:00+08 (31 minutes).\n"
    f"Top 20 component-KPIs by anomaly_count (compact):\n"
    f"- All top entries have anomaly_count = {common_count}, i.e., value ≥ global P95 for every minute in the window.\n"
    f"- Earliest anomaly timestamp in the top results: {earliest_dt} (UTC+8).\n"
    f"- Representative top items (component/kpi): {', '.join(examples_list)}\n"
    f"Observation: many top KPIs are container start times, node disk total, and JVM/runtime OS metrics — indicating these metrics were at or above their global P95 throughout the window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_P95
857                adservice2                runtime.java_lang_Runtime_StartTime             31                  1647748800         1.647263e+12  1.647263e+12
250                 adservice                runtime.java_lang_Runtime_StartTime             31                  1647748800         1.647144e+12  1.647144e+12
3643                   node-5                                  system.disk.total             31                  1647748800         1.854945e+10  1.854945e+10
3409                   node-1                                  system.disk.total             31                  1647748800         1.406688e+10  1.406688e+10
3468                   node-2                                  system.disk.total             31                  1647748800         1.406688e+10  1.406688e+10
3526                   node-3                                  system.disk.total             31                  1647748800         1.406688e+10  1.406688e+10
845                adservice2  runtime.java_lang_OperatingSystem_CommittedVir...             31                  1647748800         3.703001e+09  3.703001e+09
238                 adservice  runtime.java_lang_OperatingSystem_CommittedVir...             31                  1647748800         3.591926e+09  3.591926e+09
1401           cartservice2-0      container.node-5.container_start_time_seconds             31                  1647748800         1.647704e+09  1.647704e+09
1105            cartservice-0      container.node-6.container_start_time_seconds             31                  1647748800         1.647687e+09  1.647687e+09
1301            cartservice-2      container.node-6.container_start_time_seconds             31                  1647748800         1.647687e+09  1.647687e+09
1205            cartservice-1      container.node-6.container_start_time_seconds             31                  1647748800         1.647685e+09  1.647685e+09
3952         paymentservice-2      container.node-6.container_start_time_seconds             31                  1647748800         1.647318e+09  1.647318e+09
1003             adservice2-0      container.node-6.container_start_time_seconds             31                  1647748800         1.647263e+09  1.647263e+09
2309       currencyservice2-0      container.node-6.container_start_time_seconds             31                  1647748800         1.647236e+09  1.647236e+09
2221        currencyservice-2      container.node-6.container_start_time_seconds             31                  1647748800         1.647236e+09  1.647236e+09
3073               frontend-2      container.node-5.container_start_time_seconds             31                  1647748800         1.647144e+09  1.647144e+09
4760  recommendationservice-2      container.node-6.container_start_time_seconds             31                  1647748800         1.647144e+09  1.647144e+09
1781        checkoutservice-2      container.node-5.container_start_time_seconds             31                  1647748800         1.647144e+09  1.647144e+09
2885               frontend-1      container.node-5.container_start_time_seconds             31                  1647748800         1.647144e+09  1.647144e+09, 5253)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# reuse existing metric if available
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (reuse if present)
if 'global_p95' in globals():
    gp95 = global_p95.copy()
else:
    gp95 = (
        metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'global_P95'})
    )

# 2) Filter rows to the incident window: 2022-03-20 12:00:00+08 -> 2022-03-20 12:30:00+08 inclusive
# reuse start_ts/end_ts if present otherwise compute
if 'start_ts' in globals() and 'end_ts' in globals():
    s_ts, e_ts = start_ts, end_ts
else:
    s_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
    e_ts = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

metric_window = metric[(metric['timestamp'] >= s_ts) & (metric['timestamp'] <= e_ts)].copy()

# 3) Exclude any kpi_name that contains any of these substrings (case-sensitive)
exclude_substrings = ['StartTime', 'container_start_time', 'system.disk.total', 'Runtime_StartTime', 'LastGcInfo']
pattern = '|'.join(re.escape(s) for s in exclude_substrings)

# Determine which (cmdb_id,kpi_name) pairs to exclude based on kpi_name matching pattern
# We'll apply exclusion to both the gp95 list (to count scanned after exclusion) and to the window data
gp95_filtered = gp95[~gp95['kpi_name'].str.contains(pattern, regex=True)].copy()
total_scanned_after_exclusion = gp95_filtered.shape[0]

# Filter window rows to exclude those kpi_names
metric_window_filtered = metric_window[~metric_window['kpi_name'].str.contains(pattern, regex=True)].copy()

# 4) For each remaining (cmdb_id, kpi_name) compute required aggregates
# Merge windowed data with global P95 thresholds (use gp95, but only for remaining KPIs for efficiency)
merged = metric_window_filtered.merge(gp95_filtered, on=['cmdb_id', 'kpi_name'], how='inner')

# Mark anomalies where value >= global_P95 (threshold computed globally)
merged['is_anomaly'] = merged['value'] >= merged['global_P95']

# max_value_in_window per group (even if no anomalies)
max_vals = (
    merged
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# anomalies aggregation: count and earliest timestamp
anomalies = (
    merged[merged['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(anomaly_count=('timestamp', 'count'),
         earliest_anomaly_timestamp=('timestamp', 'min'))
)

# Combine results
results = (
    max_vals
    .merge(gp95_filtered, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anomalies, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill missing anomaly_count with 0 and keep earliest timestamp nullable
results['anomaly_count'] = results['anomaly_count'].fillna(0).astype(int)
results['earliest_anomaly_timestamp'] = results['earliest_anomaly_timestamp'].astype('Int64')

# 4) severity_ratio = (max_value_in_window - global_P95)/global_P95 with safe denom
# replace zeros in global_P95 with a tiny value to avoid div-by-zero
safe_den = results['global_P95'].replace(0, 1e-12)
results['severity_ratio'] = (results['max_value_in_window'] - results['global_P95']) / safe_den

# 5) Return compact table of top 30 rows where anomaly_count>0 sorted by severity_ratio desc
results_with_anom = results[results['anomaly_count'] > 0].copy()
top30 = results_with_anom.sort_values('severity_ratio', ascending=False).head(30)

# Keep only required columns and keep output compact
top30 = top30[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'global_P95', 'severity_ratio']]

# Display the top30 and the total number scanned after exclusion
top30, total_scanned_after_exclusion

```
Out[3]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse kernel variables
try:
    top30
    total_scanned_after_exclusion
except NameError:
    raise

# derive concise facts
earliest_ts = top30['earliest_anomaly_timestamp'].dropna().astype(int).min()
earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(earliest_ts) else None
count_full_window = int((top30['anomaly_count'] == 31).sum())
total_top30 = top30.shape[0]
zero_gp95_count = int((top30['global_P95'] == 0).sum())
examples = [f"{r[0]}/{r[1]}" for r in top30[['cmdb_id','kpi_name']].head(6).to_records(index=False)]

summary = (
    f"Summary of metric analysis (exclusions applied):\n"
    f"- Total distinct component-KPIs scanned after exclusion: {int(total_scanned_after_exclusion)}.\n"
    f"- Returned top {total_top30} rows where anomaly_count>0, sorted by severity_ratio desc.\n"
    f"- Many top entries ( {count_full_window} of {total_top30} ) had anomaly_count = 31 (value ≥ global P95 for every minute in the 12:00–12:30 window).\n"
    f"- Earliest anomaly in these top results: {earliest_dt} (UTC+8).\n"
    f"- {zero_gp95_count} of the top {total_top30} have global_P95 == 0, which yields very large computed severity_ratio (safe divisor was used to avoid divide-by-zero).\n"
    f"- Representative top items: {', '.join(examples)}.\n"
    f"- Common high-severity KPIs include container memory failure counters and container filesystem reads/writes on /dev/vda, indicating I/O and memory-related spikes during the incident window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_P95  severity_ratio
4946        shippingservice2-0  container.node-6.container_memory_failures.hie...             31                  1647748800          3828.000000  0.000000e+00    3.828000e+15
4944        shippingservice2-0  container.node-6.container_memory_failures.con...             31                  1647748800          3828.000000  0.000000e+00    3.828000e+15
4272   recommendationservice-0       container.node-6.container_fs_reads./dev/vda             31                  1647748800           931.000000  0.000000e+00    9.310000e+14
4466   recommendationservice-2       container.node-6.container_fs_reads./dev/vda             31                  1647748800           781.000000  0.000000e+00    7.810000e+14
4563  recommendationservice2-0       container.node-6.container_fs_reads./dev/vda             31                  1647748800           627.000000  0.000000e+00    6.270000e+14
...                        ...                                                ...            ...                         ...                  ...           ...             ...
4377   recommendationservice-1      container.node-6.container_fs_writes./dev/vda             31                  1647748800             8.000000  0.000000e+00    8.000000e+12
2984               frontend2-0       container.node-6.container_fs_reads./dev/vda             31                  1647748800             6.000000  0.000000e+00    6.000000e+12
2784                frontend-2  container.node-5.container_cpu_cfs_throttled_s...              1                  1647750180             0.005072  8.881784e-16    5.710063e+12
4931        shippingservice2-0    container.node-6.container_fs_reads_MB./dev/vda             31                  1647748800             5.414062  0.000000e+00    5.414062e+12
1140            cartservice2-0       container.node-5.container_fs_reads./dev/vda             31                  1647748800             5.000000  0.000000e+00    5.000000e+12

[30 rows x 7 columns], 4993)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# reuse existing 'trace' variable if present, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using the full series (no filtering)
gp = (
    trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile([0.95, 0.05])
    .unstack(level=2)
    .reset_index()
)
# After unstack the columns are 0.05 and 0.95; ensure naming
gp.columns = ['cmdb_id', 'trace_name', 'global_P05', 'global_P95']

# 2) Filter trace rows to the incident window 2022-03-20 12:00:00+08 → 2022-03-20 12:30:00+08 inclusive
# reuse start_ts/end_ts if available from prior cells, else compute
try:
    s_ts, e_ts
except NameError:
    s_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
    e_ts = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

trace_win = trace[(trace['timestamp'] >= s_ts) & (trace['timestamp'] <= e_ts)].copy()

# 3) Define anomaly types and conditions
# row_count -> low anomaly (value <= global_P05)
is_row_count = gp['trace_name'].str.contains('row_count')  # substring match
is_degradation = gp['trace_name'].str.contains('duration') | gp['trace_name'].str.contains('error_rate')

# We'll annotate gp with anomaly_type where applicable
gp['anomaly_type'] = np.where(is_row_count, 'low',
                             np.where(is_degradation, 'high', None))

# Keep only trace series that have an anomaly_type defined (others are ignored)
gp_anomalous = gp[gp['anomaly_type'].notna()].copy()

# 4) For each (cmdb_id, trace_name) compute aggregates in the window
# Merge windowed data with global thresholds but only for the anomalous trace series
merged = trace_win.merge(gp_anomalous, on=['cmdb_id', 'trace_name'], how='inner')

# Determine anomaly rows based on anomaly_type
merged['is_anomaly'] = False
# low anomalies: value <= global_P05
merged.loc[merged['anomaly_type'] == 'low', 'is_anomaly'] = (
    merged.loc[merged['anomaly_type'] == 'low', 'value'] <= merged.loc[merged['anomaly_type'] == 'low', 'global_P05']
)
# high anomalies: value >= global_P95
merged.loc[merged['anomaly_type'] == 'high', 'is_anomaly'] = (
    merged.loc[merged['anomaly_type'] == 'high', 'value'] >= merged.loc[merged['anomaly_type'] == 'high', 'global_P95']
)

# Aggregations per (cmdb_id, trace_name)
agg_all = (
    merged
    .groupby(['cmdb_id', 'trace_name', 'anomaly_type'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[merged.loc[x.index, 'is_anomaly']].min() if merged.loc[x.index, 'is_anomaly'].any() else pd.NA),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
        global_P95=('global_P95', 'first'),
        global_P05=('global_P05', 'first')
    )
)

# Ensure earliest_anomaly_timestamp is nullable Int64
agg_all['earliest_anomaly_timestamp'] = agg_all['earliest_anomaly_timestamp'].astype('Int64')

# 5A) Top 20 (cmdb_id, trace_name) with anomaly_count>0 sorted by anomaly_count desc
top_traces = agg_all[agg_all['anomaly_count'] > 0].sort_values('anomaly_count', ascending=False).head(20)
top_traces = top_traces[['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp',
                         'max_value_in_window', 'min_value_in_window', 'global_P95', 'global_P05', 'anomaly_type']]

# 5B) Aggregated per-cmdb_id summary for the window
cmdb_agg = (
    top_traces
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_anomaly_count=('anomaly_count', 'sum'),
        earliest_anomaly_timestamp=('earliest_anomaly_timestamp', lambda x: x.dropna().min() if x.notna().any() else pd.NA),
        number_of_faulty_trace_names=('trace_name', 'nunique')
    )
)
cmdb_agg['earliest_anomaly_timestamp'] = cmdb_agg['earliest_anomaly_timestamp'].astype('Int64')

# Keep outputs compact (top 20 rows each)
top_traces, cmdb_agg.head(20)

```
Out[4]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse kernel variables from previous execution
try:
    top_traces
    cmdb_agg
except NameError:
    raise

# derive concise summary facts
earliest_ts = int(top_traces['earliest_anomaly_timestamp'].dropna().min())
earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
total_faulty_traces = int(top_traces.shape[0])
high_count = int((top_traces['anomaly_type'] == 'high').sum())
low_count = int((top_traces['anomaly_type'] == 'low').sum())

# select representative examples
examples = top_traces[['cmdb_id', 'trace_name', 'anomaly_count', 'anomaly_type']].head(6).to_records(index=False)
examples_list = [f"{c}/{t} ({a}, {typ})" for c, t, a, typ in examples]

# per-cmdb_id highlights (from cmdb_agg)
cmdb_highlights = cmdb_agg.sort_values('total_anomaly_count', ascending=False).head(4)
cmdb_list = [
    f"{row['cmdb_id']}: total_anomaly_count={int(row['total_anomaly_count'])}, "
    f"faulty_trace_names={int(row['number_of_faulty_trace_names'])}, "
    f"earliest={pd.to_datetime(int(row['earliest_anomaly_timestamp']), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')}"
    for _, row in cmdb_highlights.iterrows()
]

summary = (
    "Trace analysis summary:\n"
    f"- Global P95/P5 computed per (cmdb_id, trace_name). Window: 2022-03-20 12:00:00+08 → 12:30:00+08.\n"
    f"- Earliest anomaly observed at: {earliest_dt} (UTC+8).\n"
    f"- {total_faulty_traces} distinct (cmdb_id, trace_name) had anomalies in the window (top results shown).\n"
    f"  Of these, {high_count} are 'high' (duration/error_rate degradations) and {low_count} are 'low' (row_count traffic drops).\n"
    f"- Representative anomalous traces: {', '.join(examples_list)}.\n"
    f"- Per-component highlights (top 4): { '; '.join(cmdb_list) }.\n"
    f"- Key observation: many high-severity anomalies are error_rate/duration spikes originating from frontend and checkoutservice calls to downstream services, while at least one low-severity anomaly indicates reduced traffic (row_count) to currencyservice—suggesting degradations in request handling and downstream errors during the incident window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                   trace_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_P95  global_P05 anomaly_type
850        frontend2-0                        trace.self.error_rate             31                  1647748800             0.000000             0.000000    0.000000    0.000000         high
378  checkoutservice-2         trace.to_paymentservice-0.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
376  checkoutservice-2      trace.to_paymentservice-0.duration_mean             30                  1647748800             0.000209             0.000140    0.000140    0.000185         high
374  checkoutservice-2           trace.to_emailservice-2.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
372  checkoutservice-2        trace.to_emailservice-2.duration_mean             30                  1647748800             0.000255             0.000237    0.000237    0.000254         high
722         frontend-1  trace.to_productcatalogservice-1.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
718         frontend-1  trace.to_productcatalogservice-0.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
730         frontend-1  trace.to_recommendationservice-0.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
726         frontend-1  trace.to_productcatalogservice-2.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
738         frontend-1  trace.to_recommendationservice-2.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
734         frontend-1  trace.to_recommendationservice-1.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
363  checkoutservice-2         trace.to_currencyservice-2.row_count             30                  1647748800             6.000000             2.000000    2.450000    6.000000          low
362  checkoutservice-2        trace.to_currencyservice-2.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
818         frontend-2  trace.to_productcatalogservice-2.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
778         frontend-2            trace.to_cartservice-1.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
826         frontend-2  trace.to_recommendationservice-1.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
822         frontend-2  trace.to_recommendationservice-0.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
834         frontend-2        trace.to_shippingservice-0.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high
833         frontend-2      trace.to_shippingservice-0.duration_p95             30                  1647748800             0.000128             0.000073    0.000073    0.000096         high
830         frontend-2  trace.to_recommendationservice-2.error_rate             30                  1647748800             0.000000             0.000000    0.000000    0.000000         high,              cmdb_id  total_anomaly_count  earliest_anomaly_timestamp  number_of_faulty_trace_names
0  checkoutservice-2                  180                  1647748800                             6
1         frontend-1                  180                  1647748800                             6
2         frontend-2                  210                  1647748800                             7
3        frontend2-0                   31                  1647748800                             1)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# reuse existing 'log' if present, otherwise load
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

# 1) Compute global P95 and P5 per (cmdb_id, log_name) using the full series (no filtering)
gp = (
    log
    .groupby(['cmdb_id', 'log_name'])['value']
    .quantile([0.95, 0.05])
    .unstack(level=2)
    .reset_index()
)
# Ensure columns present and named
gp.columns = ['cmdb_id', 'log_name', 'global_P05', 'global_P95']

# 2) Filter log rows to the incident window: reuse start_ts/end_ts if available
try:
    s_ts, e_ts
except NameError:
    s_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
    e_ts = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

log_win = log[(log['timestamp'] >= s_ts) & (log['timestamp'] <= e_ts)].copy()

# 3) Define anomalies:
# For 'log.error_count' anomaly = value >= global_P95 (high)
# For 'log.row_count' anomaly = value <= global_P05 (low)
# Keep only these two log_names for analysis
gp_focus = gp[gp['log_name'].isin(['log.error_count', 'log.row_count'])].copy()

# 4) Merge windowed data with global thresholds and compute anomalies row-wise
merged = log_win.merge(gp_focus, on=['cmdb_id', 'log_name'], how='inner')

# Initialize
merged['anomaly_type'] = np.where(merged['log_name'] == 'log.error_count', 'high',
                                 np.where(merged['log_name'] == 'log.row_count', 'low', None))
# Determine anomaly boolean
merged['is_anomaly'] = False
merged.loc[merged['anomaly_type'] == 'high', 'is_anomaly'] = (
    merged.loc[merged['anomaly_type'] == 'high', 'value'] >= merged.loc[merged['anomaly_type'] == 'high', 'global_P95']
)
merged.loc[merged['anomaly_type'] == 'low', 'is_anomaly'] = (
    merged.loc[merged['anomaly_type'] == 'low', 'value'] <= merged.loc[merged['anomaly_type'] == 'low', 'global_P05']
)

# Aggregations per (cmdb_id, log_name)
agg_all = (
    merged
    .groupby(['cmdb_id', 'log_name', 'anomaly_type'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[merged.loc[x.index, 'is_anomaly']].min() if merged.loc[x.index, 'is_anomaly'].any() else pd.NA),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
        global_P95=('global_P95', 'first'),
        global_P05=('global_P05', 'first')
    )
)

agg_all['earliest_anomaly_timestamp'] = agg_all['earliest_anomaly_timestamp'].astype('Int64')
# Keep only rows with anomaly_count > 0
agg_anom = agg_all[agg_all['anomaly_count'] > 0].copy()

# 5A) Top 20 rows sorted by anomaly_count desc
top_logs = agg_anom.sort_values('anomaly_count', ascending=False).head(20)
top_logs = top_logs[['cmdb_id', 'log_name', 'anomaly_count', 'earliest_anomaly_timestamp',
                     'max_value_in_window', 'min_value_in_window', 'global_P95', 'global_P05', 'anomaly_type']]

# 5B) Aggregated per-cmdb_id summary for the window
cmdb_log_agg = (
    top_logs
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_anomaly_count=('anomaly_count', 'sum'),
        earliest_anomaly_timestamp=('earliest_anomaly_timestamp', lambda x: x.dropna().min() if x.notna().any() else pd.NA),
        number_of_faulty_log_names=('log_name', 'nunique')
    )
)
cmdb_log_agg['earliest_anomaly_timestamp'] = cmdb_log_agg['earliest_anomaly_timestamp'].astype('Int64')

# Return compact outputs (limited rows)
top_logs, cmdb_log_agg.head(10)

```
Out[5]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse kernel variables produced earlier
try:
    top_logs
    cmdb_log_agg
except NameError:
    raise

n_faulty_rows = int(top_logs.shape[0])
n_faulty_cmdb = int(cmdb_log_agg.shape[0])
earliest_ts = int(top_logs['earliest_anomaly_timestamp'].dropna().min())
earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
error_count_rows = int((top_logs['log_name'] == 'log.error_count').sum())
row_count_rows = int((top_logs['log_name'] == 'log.row_count').sum())
zero_p95_count = int((top_logs['global_P95'] == 0).sum())

summary = (
    f"Log analysis summary:\n"
    f"- Found {n_faulty_rows} (cmdb_id, log_name) entries with anomalies in the incident window; these span {n_faulty_cmdb} components.\n"
    f"- Earliest anomaly observed at: {earliest_dt} (UTC+8).\n"
    f"- Most anomalies are for 'log.error_count' (count={error_count_rows}); many of these have global_P95 == 0 ({zero_p95_count} entries),\n"
    f"  meaning the threshold is zero and every zero-valued sample meets value >= P95 — this likely produces false-positive flags and should be interpreted cautiously.\n"
    f"- One notable 'low' anomaly is 'log.row_count' for recommendationservice-2 (row_count anomaly_count=31) showing traffic variation (window min=3, max=252).\n"
    f"- Per-component aggregation (top results) shows many services (e.g., adservice-2, cartservice-*, checkoutservice-2, currencyservice-*) each with anomaly_count=31 starting at 2022-03-20 12:00:00+08.\n"
    f"Recommendation: investigate the meaning of zero thresholds for log.error_count (possible metric sparsity or quantile artifact) and prioritize components with non-trivial value changes (e.g., row_count drops) for operational follow-up."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id         log_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_P95  global_P05 anomaly_type
4               adservice-2  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
6             cartservice-0  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
8             cartservice-1  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
10            cartservice-2  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
18        currencyservice-0  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
16        checkoutservice-2  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
22        currencyservice-2  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
20        currencyservice-1  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
52  recommendationservice-2  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
50  recommendationservice-1  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
42  productcatalogservice-0  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
36         paymentservice-0  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
34               frontend-2  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
30               frontend-0  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
48  recommendationservice-0  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
53  recommendationservice-2    log.row_count             31                  1647748800                252.0                  3.0       212.0       252.0          low
56        shippingservice-0  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
60        shippingservice-2  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
58        shippingservice-1  log.error_count             31                  1647748800                  0.0                  0.0         0.0         0.0         high
46  productcatalogservice-2  log.error_count             30                  1647748800                  0.0                  0.0         0.0         0.0         high,              cmdb_id  total_anomaly_count  earliest_anomaly_timestamp  number_of_faulty_log_names
0        adservice-2                   31                  1647748800                           1
1      cartservice-0                   31                  1647748800                           1
2      cartservice-1                   31                  1647748800                           1
3      cartservice-2                   31                  1647748800                           1
4  checkoutservice-2                   31                  1647748800                           1
5  currencyservice-0                   31                  1647748800                           1
6  currencyservice-1                   31                  1647748800                           1
7  currencyservice-2                   31                  1647748800                           1
8         frontend-0                   31                  1647748800                           1
9         frontend-2                   31                  1647748800                           1)```
```

In [ ]:
# In[6]:

import pandas as pd
import re
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# reuse existing metric and global_p95 if available
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (reuse if exists)
if 'global_p95' in globals():
    gp95 = global_p95.copy()
else:
    gp95 = (
        metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'global_P95'})
    )

# 2) Filter metric rows to the incident window 2022-03-20 12:00:00+08 → 2022-03-20 12:30:00+08
try:
    s_ts, e_ts
except NameError:
    s_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
    e_ts = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

metric_window = metric[(metric['timestamp'] >= s_ts) & (metric['timestamp'] <= e_ts)].copy()

# 3) Select metric rows whose kpi_name contains any of the specified substrings
substrs = ['container_fs_reads', 'container_fs_writes', 'container_fs_reads_MB', 'container_fs_writes_MB', 'container_memory_failures']
pattern = '|'.join(re.escape(s) for s in substrs)
selected = metric_window[metric_window['kpi_name'].str.contains(pattern, regex=True)].copy()

# 4) Merge with precomputed global P95 (must be computed from full series) and mark anomalies (value >= global_P95)
selected = selected.merge(gp95, on=['cmdb_id', 'kpi_name'], how='left')

# safe comparison: treat NaN global_P95 as 0 to avoid dropping rows (but note this means >=0 may mark anomalies)
selected['global_P95'] = selected['global_P95'].fillna(0.0)
selected['is_anomaly'] = selected['value'] >= selected['global_P95']

# 5) Map each row to node identifier:
#   - if kpi_name contains 'container.node-<N>' extract that and map to 'node-<N>'
#   - else if cmdb_id starts with 'node-' use cmdb_id
def extract_node_id(row):
    m = re.search(r'container\.node-(\d+)', row['kpi_name'])
    if m:
        return f"node-{m.group(1)}"
    if isinstance(row['cmdb_id'], str) and row['cmdb_id'].startswith('node-'):
        return row['cmdb_id']
    return 'unknown'

selected['node_id'] = selected.apply(extract_node_id, axis=1)

# Keep only anomalous rows for aggregation
anoms = selected[selected['is_anomaly']].copy()

# If no anomalies, prepare an empty compact result
if anoms.shape[0] == 0:
    nodes_summary = pd.DataFrame(columns=[
        'node_id', 'total_anomaly_count', 'earliest_anomaly_timestamp',
        'number_of_affected_containers', 'top_affected_containers'
    ])
    nodes_scanned = 0
else:
    # total_anomaly_count per node
    node_counts = anoms.groupby('node_id').agg(
        total_anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        number_of_affected_containers=('cmdb_id', lambda x: x.nunique())
    ).reset_index()

    # top affected containers per node: compute anomaly counts per (node,cmdb_id)
    cont_counts = anoms.groupby(['node_id', 'cmdb_id']).size().reset_index(name='cnt')

    # For each node, get top up to 10 cmdb_id sorted by cnt desc
    top_containers = (
        cont_counts
        .sort_values(['node_id', 'cnt', 'cmdb_id'], ascending=[True, False, True])
        .groupby('node_id')
        .agg(top_affected_containers=('cmdb_id', lambda x: ','.join(x.head(10))))
        .reset_index()
    )

    # Merge into node_counts
    nodes_summary = node_counts.merge(top_containers, on='node_id', how='left')

    # Sort by total_anomaly_count desc and limit to top 10
    nodes_summary = nodes_summary.sort_values('total_anomaly_count', ascending=False).head(10)

    nodes_scanned = selected['node_id'].nunique()

# Keep output compact
nodes_summary = nodes_summary[['node_id', 'total_anomaly_count', 'earliest_anomaly_timestamp',
                               'number_of_affected_containers', 'top_affected_containers']]

# Display compact result and number of nodes scanned (after selection)
nodes_summary, nodes_scanned

```
Out[6]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse kernel variables nodes_summary and nodes_scanned from previous cell
try:
    nodes_summary
    nodes_scanned
except NameError:
    raise

# build concise plain-English summary
rows = []
for _, r in nodes_summary.iterrows():
    node = r['node_id']
    count = int(r['total_anomaly_count'])
    earliest_ts = int(r['earliest_anomaly_timestamp'])
    earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    affected = int(r['number_of_affected_containers'])
    top_list = (r['top_affected_containers'] or '').split(',')[:10]
    top_preview = ','.join(top_list)
    rows.append(f"- {node}: total_anomaly_count={count}, earliest={earliest_dt}, affected_containers={affected}, top_containers=[{top_preview}]")

summary = (
    f"Focused metric I/O/memory failure analysis (window 2022-03-20 12:00–12:30 UTC+8):\n"
    f"- Nodes scanned (from selected KPIs): {int(nodes_scanned)}.\n"
    f"- Top nodes by anomaly count:\n" + "\n".join(rows)
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(  node_id  total_anomaly_count  earliest_anomaly_timestamp  number_of_affected_containers                            top_affected_containers
1  node-6                11588                  1647748800                             34  checkoutservice-0,checkoutservice-1,checkoutse...
0  node-5                 1762                  1647748800                              6  cartservice2-0,frontend-1,adservice-2,frontend..., 2)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# reuse existing metric and gp95/global_p95 if present
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# 1) Ensure global P95 per (cmdb_id, kpi_name) computed from the full series (no filtering)
if 'global_p95' in globals():
    gp95 = global_p95.copy()
elif 'gp95' in globals():
    gp95 = gp95.copy()
else:
    gp95 = (
        metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'global_P95'})
    )

# 2) Filter metric rows to the incident window 2022-03-20 12:00:00+08 → 2022-03-20 12:30:00+08 (reuse s_ts/e_ts if available)
try:
    s_ts, e_ts
except NameError:
    s_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
    e_ts = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

metric_window = metric[(metric['timestamp'] >= s_ts) & (metric['timestamp'] <= e_ts)].copy()

# 3) Select rows for cmdb_id 'currencyservice-2' and kpi_name containing any specified substrings
target_cmdb = 'currencyservice-2'
substrs = ['container.node-', 'container_fs_reads', 'container_fs_writes',
           'container_fs_reads_MB', 'container_fs_writes_MB', 'container_memory_failures']
pattern = '|'.join(re.escape(s) for s in substrs)

selected = metric_window[
    (metric_window['cmdb_id'] == target_cmdb) &
    (metric_window['kpi_name'].str.contains(pattern, regex=True))
].copy()

# If no selected rows, return explicit empty indicator
if selected.shape[0] == 0:
    empty_result = "Empty result: no matching KPIs for currencyservice-2 in the incident window."
    empty_result
else:
    # 4) Merge with global P95 computed from full series (do not recompute after filtering)
    selected = selected.merge(gp95, on=['cmdb_id', 'kpi_name'], how='left')

    # Drop rows without a global_P95 (no full-series threshold available)
    selected = selected[~selected['global_P95'].isna()].copy()

    if selected.shape[0] == 0:
        empty_result = "Empty result: matching KPIs exist in window but no global P95 found for them."
        empty_result
    else:
        # Determine anomalies: value >= global_P95 (safe comparison)
        selected['is_anomaly'] = selected['value'] >= selected['global_P95']

        # 5) Extract node_id from kpi_name substring 'container.node-<N>' if present, else 'unknown'
        def extract_node(kname):
            m = re.search(r'container\.node-(\d+)', kname)
            return f"node-{m.group(1)}" if m else 'unknown'

        selected['node_id'] = selected['kpi_name'].apply(extract_node)

        # Keep only anomalous rows for aggregation
        anoms = selected[selected['is_anomaly']].copy()

        if anoms.shape[0] == 0:
            empty_result = "Empty result: no anomalies (value >= global_P95) for the selected KPIs of currencyservice-2 in the window."
            empty_result
        else:
            # Aggregate per (cmdb_id, kpi_name)
            agg = (
                anoms
                .groupby(['cmdb_id', 'kpi_name', 'node_id'], as_index=False)
                .agg(
                    anomaly_count=('is_anomaly', 'sum'),
                    earliest_anomaly_timestamp=('timestamp', 'min'),
                    max_value_in_window=('value', 'max'),
                    global_P95=('global_P95', 'first')
                )
            )

            # Ensure compactness and ordering
            result = agg[['cmdb_id', 'kpi_name', 'node_id', 'anomaly_count',
                          'earliest_anomaly_timestamp', 'max_value_in_window', 'global_P95']]
            result = result.sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20)

            # Display compact result and the number of matched rows
            result, result.shape[0]

```
Out[7]:
```
import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Load metric if not in kernel
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# 1) Global P95 per (cmdb_id, kpi_name) from full series
gp95 = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# 2) Incident window timestamps (UTC+8)
s_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
e_ts = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

metric_window = metric[(metric['timestamp'] >= s_ts) & (metric['timestamp'] <= e_ts)].copy()

# 3) Select rows for currencyservice-2 and matching kpi_name substrings
target_cmdb = 'currencyservice-2'
substrs = ['container.node-', 'container_fs_reads', 'container_fs_writes',
           'container_fs_reads_MB', 'container_fs_writes_MB', 'container_memory_failures']
pattern = '|'.join(re.escape(s) for s in substrs)

selected = metric_window[
    (metric_window['cmdb_id'] == target_cmdb) &
    (metric_window['kpi_name'].str.contains(pattern, regex=True))
].copy()

# Prepare summary logic
if selected.shape[0] == 0:
    summary = "Empty result: no matching KPIs for currencyservice-2 in the incident window."
else:
    # 4) Merge with global P95 (from full series) and determine anomalies (value >= global_P95)
    selected = selected.merge(gp95, on=['cmdb_id', 'kpi_name'], how='left')
    # Drop rows without a global_P95
    selected = selected[~selected['global_P95'].isna()].copy()
    if selected.shape[0] == 0:
        summary = "Empty result: matching KPIs exist in window but no global P95 found for them."
    else:
        selected['is_anomaly'] = selected['value'] >= selected['global_P95']
        # 4b) extract node_id from kpi_name
        def extract_node(kname):
            m = re.search(r'container\.node-(\d+)', kname)
            return f"node-{m.group(1)}" if m else 'unknown'
        selected['node_id'] = selected['kpi_name'].apply(extract_node)

        anoms = selected[selected['is_anomaly']].copy()
        if anoms.shape[0] == 0:
            summary = "Empty result: no anomalies (value >= global_P95) for the selected KPIs of currencyservice-2 in the window."
        else:
            # 5) Aggregate per (cmdb_id, kpi_name)
            agg = (
                anoms
                .groupby(['cmdb_id', 'kpi_name', 'node_id'], as_index=False)
                .agg(
                    anomaly_count=('is_anomaly', 'sum'),
                    earliest_anomaly_timestamp=('timestamp', 'min'),
                    max_value_in_window=('value', 'max'),
                    global_P95=('global_P95', 'first')
                )
            )
            agg = agg.sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20)

            # Build concise plain-English summary
            total_kpis = int(agg.shape[0])
            total_anomalous_rows = int(anoms.shape[0])
            earliest_ts = int(agg['earliest_anomaly_timestamp'].min())
            earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            # top KPI examples
            examples = agg[['kpi_name','node_id','anomaly_count']].head(5).to_records(index=False)
            examples_list = [f"{k} (node={n}, count={c})" for k,n,c in examples]

            summary = (
                f"currencyservice-2 analysis (window 2022-03-20 12:00–12:30 UTC+8): "
                f"Found {total_kpis} KPI(s) with anomalies, {total_anomalous_rows} anomalous row(s) total. "
                f"Earliest anomaly at {earliest_dt}. Top examples: {', '.join(examples_list)}."
            )

# Output concise summary
summary

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[8]:

import pandas as pd
import re
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# reuse metric if already loaded, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure timestamp column is integer type
metric['timestamp'] = metric['timestamp'].astype('int64')

# --- Step 1: find kpi_names for currencyservice-2 containing 'container.node-<N>' and map to node IDs ---
cmdb_target = 'currencyservice-2'
mask_node_kpis = metric['cmdb_id'].eq(cmdb_target) & metric['kpi_name'].str.contains(r'container\.node-\d+', regex=True)

kpi_node_df = metric.loc[mask_node_kpis, ['kpi_name']].drop_duplicates().copy()
# extract node_id
kpi_node_df['node_id'] = kpi_node_df['kpi_name'].str.extract(r'container\.node-(\d+)').fillna('').astype(str)
kpi_node_df['node_id'] = kpi_node_df['node_id'].apply(lambda x: f"node-{x}" if x and x != 'nan' else 'unknown')

# count distinct kpi_names per node_id
node_kpi_counts = (
    kpi_node_df.groupby('node_id', as_index=False)
    .agg(distinct_kpi_names=('kpi_name', 'nunique'))
    .sort_values('distinct_kpi_names', ascending=False)
)

# limit to top 10
node_kpi_counts_top10 = node_kpi_counts.head(10)

# --- Step 2: compute global P95 for selected KPI substrings (full series) for currencyservice-2 ---
substrs = ['container_fs_reads', 'container_fs_writes', 'container_fs_reads_MB', 'container_fs_writes_MB', 'container_memory_failures']
pattern = '|'.join(re.escape(s) for s in substrs)

# identify all matching kpi_names (full series) for the target cmdb
mask_selected_kpis_full = metric['cmdb_id'].eq(cmdb_target) & metric['kpi_name'].str.contains(pattern, regex=True)
selected_kpis_full = metric.loc[mask_selected_kpis_full, ['cmdb_id', 'kpi_name']].drop_duplicates().copy()

if selected_kpis_full.shape[0] == 0:
    gp95_table = pd.DataFrame({'info': ["Empty result: no matching KPIs found for currencyservice-2 in full series."]})
else:
    # compute global P95 across the full metric series (no filtering) for these (cmdb_id,kpi_name)
    gp95_all = (
        metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'global_P95'})
    )
    gp95_selected = gp95_all.merge(selected_kpis_full, on=['cmdb_id', 'kpi_name'], how='inner')
    # extract node_id from kpi_name if present
    gp95_selected['node_id'] = gp95_selected['kpi_name'].str.extract(r'container\.node-(\d+)')[0].fillna('').apply(lambda x: f"node-{x}" if x != '' else 'unknown')
    # compact table
    gp95_table = gp95_selected[['cmdb_id', 'kpi_name', 'node_id', 'global_P95']].sort_values(['node_id','kpi_name']).head(20)

# --- Step 3: filter to incident window and compute anomaly counts relative to global_P95 ---
# incident window in UTC+8
start_ts = int(pd.Timestamp('2022-03-20 12:00:00', tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp('2022-03-20 12:30:00', tz=tz).tz_convert('UTC').timestamp())

metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

if 'gp95_selected' not in locals() or gp95_selected.shape[0] == 0:
    anomalies_table = pd.DataFrame({'info': ["Empty result: no selected KPIs to evaluate anomalies (see gp95_table)."]})
else:
    # merge windowed rows with global P95 for only the selected KPIs
    window_selected = metric_window.merge(gp95_selected[['cmdb_id','kpi_name','global_P95']], on=['cmdb_id','kpi_name'], how='inner')
    if window_selected.shape[0] == 0:
        anomalies_table = pd.DataFrame({'info': ["Empty result: no window rows for the selected KPIs."]})
    else:
        # mark anomalies (value >= global_P95). Use safe numeric comparison (NaNs handled)
        window_selected['global_P95'] = window_selected['global_P95'].fillna(0.0)
        window_selected['is_anomaly'] = window_selected['value'] >= window_selected['global_P95']

        # compute max value in window per KPI
        max_vals = window_selected.groupby(['cmdb_id','kpi_name'], as_index=False).agg(max_value_in_window=('value','max'))

        # compute anomaly aggregates
        anom_agg = (
            window_selected[window_selected['is_anomaly']]
            .groupby(['cmdb_id','kpi_name'], as_index=False)
            .agg(anomaly_count=('is_anomaly','sum'),
                 earliest_anomaly_timestamp=('timestamp','min'))
        )

        # combine with global P95 and node mapping
        combined = gp95_selected[['cmdb_id','kpi_name','node_id','global_P95']].merge(max_vals, on=['cmdb_id','kpi_name'], how='left').merge(anom_agg, on=['cmdb_id','kpi_name'], how='left')
        combined['anomaly_count'] = combined['anomaly_count'].fillna(0).astype(int)
        combined['earliest_anomaly_timestamp'] = combined['earliest_anomaly_timestamp'].astype('Int64')
        # keep only KPIs with anomaly_count>0 as requested
        combined_with_anoms = combined[combined['anomaly_count'] > 0].copy()

        if combined_with_anoms.shape[0] == 0:
            anomalies_table = pd.DataFrame({'info': ["No anomalies (value >= global_P95) found in the incident window for the selected KPIs."]})
        else:
            anomalies_table = combined_with_anoms[['cmdb_id','kpi_name','node_id','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_P95']].sort_values(['anomaly_count','max_value_in_window'], ascending=[False,False]).head(20)

# Final compact outputs:
#  - node_kpi_counts_top10: mapping of node_id -> count of distinct kpi_names (top 10)
#  - gp95_table: global P95 table for selected KPIs (or an empty-info DataFrame)
#  - anomalies_table: anomaly summary in window for those KPIs (or an empty-info DataFrame)
node_kpi_counts_top10, gp95_table, anomalies_table

```
Out[8]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

earliest_ts = 1647749040
earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

summary = (
    "Summary for currencyservice-2 (full-series scan + incident window):\n"
    "- Mapping: All container kpis for currencyservice-2 containing 'container.node-<N>' map to node-6. "
    "There are 64 distinct kpi_names mapped to node-6 (top mapping result).\n"
    "- Global P95 (full series): Most selected FS/read/write KPIs have global_P95 == 0.0 (quantile result), "
    "but some container_memory_failures KPIs have non-zero P95 (example: global_P95 = 3432.0).\n"
    "- Window anomalies (2022-03-20 12:00 → 12:30 UTC+8): Many KPIs show anomaly_count = 31 because their global_P95 == 0.0 "
    "(every sample in the window meets value >= 0 — interpret cautiously).\n"
    "- Notable real spike: two container_memory_failures KPIs show anomaly_count = 3 with max_value_in_window = 4306.5 "
    "and global_P95 = 3432.0; earliest of these anomalies occurred at " + earliest_dt + " (UTC+8).\n"
    "- Actionable pointer: investigate the container_memory_failures spikes on node-6 around 12:04 (UTC+8). "
    "Also review why many FS metrics have P95 == 0.0 (possible sparsity/quantile artifact) to avoid false positives."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(  node_id  distinct_kpi_names
0  node-6                  64,               cmdb_id                                           kpi_name node_id  global_P95
0   currencyservice-2       container.node-6.container_fs_reads./dev/vda  node-6         0.0
1   currencyservice-2      container.node-6.container_fs_reads./dev/vda1  node-6         0.0
2   currencyservice-2    container.node-6.container_fs_reads_MB./dev/vda  node-6         0.0
3   currencyservice-2  container.node-6.container_fs_reads_merged./de...  node-6         0.0
4   currencyservice-2      container.node-6.container_fs_writes./dev/vda  node-6         0.0
5   currencyservice-2     container.node-6.container_fs_writes./dev/vda1  node-6         0.0
6   currencyservice-2   container.node-6.container_fs_writes_MB./dev/vda  node-6         0.0
7   currencyservice-2  container.node-6.container_fs_writes_merged./d...  node-6         0.0
8   currencyservice-2  container.node-6.container_memory_failures.con...  node-6      3432.0
9   currencyservice-2  container.node-6.container_memory_failures.con...  node-6         0.0
10  currencyservice-2  container.node-6.container_memory_failures.hie...  node-6      3432.0
11  currencyservice-2  container.node-6.container_memory_failures.hie...  node-6         0.0,               cmdb_id                                           kpi_name node_id  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  global_P95
0   currencyservice-2       container.node-6.container_fs_reads./dev/vda  node-6             31                  1647748800                  0.0         0.0
1   currencyservice-2      container.node-6.container_fs_reads./dev/vda1  node-6             31                  1647748800                  0.0         0.0
2   currencyservice-2    container.node-6.container_fs_reads_MB./dev/vda  node-6             31                  1647748800                  0.0         0.0
3   currencyservice-2  container.node-6.container_fs_reads_merged./de...  node-6             31                  1647748800                  0.0         0.0
4   currencyservice-2      container.node-6.container_fs_writes./dev/vda  node-6             31                  1647748800                  0.0         0.0
5   currencyservice-2     container.node-6.container_fs_writes./dev/vda1  node-6             31                  1647748800                  0.0         0.0
6   currencyservice-2   container.node-6.container_fs_writes_MB./dev/vda  node-6             31                  1647748800                  0.0         0.0
7   currencyservice-2  container.node-6.container_fs_writes_merged./d...  node-6             31                  1647748800                  0.0         0.0
9   currencyservice-2  container.node-6.container_memory_failures.con...  node-6             31                  1647748800                  0.0         0.0
11  currencyservice-2  container.node-6.container_memory_failures.hie...  node-6             31                  1647748800                  0.0         0.0
8   currencyservice-2  container.node-6.container_memory_failures.con...  node-6              3                  1647749040               4306.5      3432.0
10  currencyservice-2  container.node-6.container_memory_failures.hie...  node-6              3                  1647749040               4306.5      3432.0)```
```